> **Author:** Brendan OConnell  
> **Year:** 2026  
> **Purpose:**   
> **Key decisions:**  
> *   
> * 

In [83]:
import pandas as pd

In [119]:
# Renormalizing function for elements + oxygen
def renormalize_with_oxygen(df_copy, non_o_element_cols, oxygen_col='o'):

    df_ret = df_copy
    scale = (100 - df_ret[oxygen_col]) / 100.0
    df_ret[non_o_element_cols] = df_ret[non_o_element_cols].multiply(scale, axis=0)
    return df_ret

In [109]:
df = pd.read_parquet("../../../data/processed/nist_concatenated_parquets/nist_neQuantC3s JORS[GSR+Base]_concatenated_data.parquet")
df.shape

(268318, 105)

In [110]:
df.columns

Index(['NUMBER', 'FIELD', 'MAGFIELD', 'XABS', 'YABS', 'XFERET', 'YFERET',
       'DAVG', 'DMAX', 'DMIN',
       ...
       'U_TM_', 'AU', 'U_AU_', 'PB', 'U_PB_', 'BI', 'U_BI_', 'sample_source',
       'sum_without_oxygen', 'sum_with_oxygen'],
      dtype='str', length=105)

In [111]:
# Drop the meta fields related to imaging measurements/configurations
df.columns[1:17]

Index(['FIELD', 'MAGFIELD', 'XABS', 'YABS', 'XFERET', 'YFERET', 'DAVG', 'DMAX',
       'DMIN', 'DPERP', 'ASPECT', 'AREA', 'PERIMETER', 'ORIENTATION', 'MAG',
       'VIDEO'],
      dtype='str')

In [112]:
cols_to_drop = list(df.columns[1:17]) + [c for c in df.columns if c.startswith('U_')]
df = df.drop(columns=cols_to_drop + ['sum_without_oxygen', 'sum_with_oxygen'])
df.columns

Index(['NUMBER', 'CLASS', 'LIVETIME', 'FIRSTELM', 'FIRSTPCT', 'SECONDELM',
       'SECONDPCT', 'THIRDELM', 'THIRDPCT', 'FOURTHELM', 'FOURTHPCT', 'COUNTS',
       'O', 'F', 'MG', 'AL', 'SI', 'P', 'S', 'CL', 'K', 'CA', 'TI', 'CR', 'MN',
       'FE', 'NI', 'CU', 'ZN', 'RB', 'SR', 'ZR', 'MO', 'RH', 'PD', 'AG', 'IN',
       'SN', 'SB', 'BA', 'LA', 'CE', 'ND', 'SM', 'TB', 'TM', 'AU', 'PB', 'BI',
       'sample_source'],
      dtype='str')

In [113]:
# Drop the rest of the noise
df.columns[2:12]

Index(['LIVETIME', 'FIRSTELM', 'FIRSTPCT', 'SECONDELM', 'SECONDPCT',
       'THIRDELM', 'THIRDPCT', 'FOURTHELM', 'FOURTHPCT', 'COUNTS'],
      dtype='str')

In [114]:
cols_to_drop = list(df.columns[2:12])
df = df.drop(columns=cols_to_drop)
df.columns

Index(['NUMBER', 'CLASS', 'O', 'F', 'MG', 'AL', 'SI', 'P', 'S', 'CL', 'K',
       'CA', 'TI', 'CR', 'MN', 'FE', 'NI', 'CU', 'ZN', 'RB', 'SR', 'ZR', 'MO',
       'RH', 'PD', 'AG', 'IN', 'SN', 'SB', 'BA', 'LA', 'CE', 'ND', 'SM', 'TB',
       'TM', 'AU', 'PB', 'BI', 'sample_source'],
      dtype='str')

In [115]:
# meta columns --- rename to match NFI dataset
df.rename(columns={"sample_source": "stub_id", "NUMBER": "particle_id"}, inplace=True)
df.columns = df.columns.str.lower()
meta_cols = ["stub_id", "particle_id", "class"]

# element columns
element_cols = [c for c in df.columns if c not in meta_cols]
gsr_cols = ["pb" , "sb", "ba"]
non_gsr_cols = sorted([c for c in element_cols if c not in gsr_cols])

# Sort the columns so they're GSR-analysis-friendly
df = df[meta_cols + gsr_cols + non_gsr_cols]
df.columns

Index(['stub_id', 'particle_id', 'class', 'pb', 'sb', 'ba', 'ag', 'al', 'au',
       'bi', 'ca', 'ce', 'cl', 'cr', 'cu', 'f', 'fe', 'in', 'k', 'la', 'mg',
       'mn', 'mo', 'nd', 'ni', 'o', 'p', 'pd', 'rb', 'rh', 's', 'si', 'sm',
       'sn', 'sr', 'tb', 'ti', 'tm', 'zn', 'zr'],
      dtype='str')

In [116]:
df.shape

(268318, 40)

Renormalize that Oooo

In [ ]:
non_o_elements = [c for c in element_cols if c != "o"]
assert len(non_o_elements) == len(element_cols) - 1

In [126]:
normalized_df = renormalize_with_oxygen(df.copy(), non_o_elements)

In [127]:
# sanity check
(normalized_df[element_cols].sum(axis=1)).describe()

count    2.683180e+05
mean     1.000000e+02
std      1.684108e-14
min      1.000000e+02
25%      1.000000e+02
50%      1.000000e+02
75%      1.000000e+02
max      1.000000e+02
dtype: float64

In [128]:
normalized_df[element_cols].sum(axis=1)

0         100.0
1         100.0
2         100.0
3         100.0
4         100.0
          ...  
268313    100.0
268314    100.0
268315    100.0
268316    100.0
268317    100.0
Length: 268318, dtype: float64

In [129]:
normalized_df.columns

Index(['stub_id', 'particle_id', 'class', 'pb', 'sb', 'ba', 'ag', 'al', 'au',
       'bi', 'ca', 'ce', 'cl', 'cr', 'cu', 'f', 'fe', 'in', 'k', 'la', 'mg',
       'mn', 'mo', 'nd', 'ni', 'o', 'p', 'pd', 'rb', 'rh', 's', 'si', 'sm',
       'sn', 'sr', 'tb', 'ti', 'tm', 'zn', 'zr'],
      dtype='str')

In [ ]:
# [auto-commented] normalized_df.to_parquet(("../../../data/processed/preprocessed_nist.parquet"))